# Pharmacology landscape analysis

We’ll analyse an experimental activity landscape to explore how chemical building-block choices affect enzyme inhibition. A combinatorial cyanimide-library screen will serve as our example, taking us from assay measurements to local optima, ruggedness and interactions.

## 1. Importing necessary dependencies

Install GraphFLA with pip, then import the dependencies below.

In [ ]:
!pip install graphfla

In [1]:
import graphfla
from graphfla.landscape import Landscape
import pandas as pd

## 2. Loading the experimental dataset

For our example, we use the cyanimide-library screen from [Kooij et al. (2025)](https://doi.org/10.1002/anie.202510941). Compounds were assembled by coupling amine building blocks with carboxylic acids, then screened as unpurified reaction mixtures against a panel of proteases.

We’ll focus on inhibition of **mouse USP18 (`mUSP18`)**. The CSV contains 7,536 assay rows, representing 7,504 distinct combinations of 16 amine blocks and 469 carboxylic-acid blocks, with some repeated measurements. This objective measures activity on one enzyme; selectivity across the panel is a separate question.

| Column | Meaning |
|---|---|
| `compound_id` | Identifier of the intended coupling product; repeated measurements share an ID. |
| `bb` | Cyanimide-bearing amine building-block identifier. |
| `ca` | Carboxylic-acid building-block identifier. |
| `mUSP18` | Percentage inhibition of mouse USP18 in the primary screen; larger is better. |

These are biochemical screening readouts from crude reaction mixtures. Let’s load the relevant columns.

In [2]:
data_url = (
    "https://raw.githubusercontent.com/COLA-Laboratory/GraphFLA/"
    "main/data/Pharmacology/DubHts/dub_hts_wide.csv"
)
df = pd.read_csv(data_url)[["compound_id", "bb", "ca", "mUSP18"]]
df.head()

,compound_id,bb,ca,mUSP18
0,BB01CA90,BB01,CA090,27.082280
1,BB01CA91,BB01,CA091,13.363539
2,BB01CA92,BB01,CA092,24.569932
3,BB01CA93,BB01,CA093,74.140837
4,BB01CA94,BB01,CA094,24.059166


## 3. Preparing the inputs

To construct a landscape, GraphFLA only needs two inputs—much like the features `X` and targets `y` used to train a machine-learning model:

- `X`: the experimental configurations, as a DataFrame or a two-dimensional NumPy array.
- `f`: one measured outcome per configuration, as a Series, list or one-dimensional array.

The rows of `X` and `f` must refer to the same configurations in the same order.


We need one score per building-block combination, so we first average repeated measurements of the same `bb`–`ca` pair. We retain the reported score scale, including negative inhibition readings near the assay baseline.

In [3]:
df = df.groupby(["bb", "ca"], as_index=False)["mUSP18"].mean()
df.head()

,bb,ca,mUSP18
0,BB01,CA001,21.663005
1,BB01,CA002,21.622812
2,BB01,CA003,22.786787
3,BB01,CA004,26.376782
4,BB01,CA005,20.430703


Now we can select the design variables and their corresponding scores:

In [4]:
X = df[['bb', 'ca']]
f = df['mUSP18']
X.head()

,bb,ca
0,BB01,CA001
1,BB01,CA002
2,BB01,CA003
3,BB01,CA004
4,BB01,CA005


## 4. Constructing the landscape

We can use `Landscape` with `bb` and `ca` declared categorical. Their numeric-looking labels are identifiers, not ordered doses or chemical distances. Neighbouring compounds share one building block and differ in the other.

We set `maximize=True` to find combinations with stronger mUSP18 inhibition.

In [5]:
landscape = Landscape(maximize=True)
landscape.build_from_data(
    X, f,
    data_types={"bb": "categorical", "ca": "categorical"},
)

Building Landscape from data...


 - Validating data types dictionary...


   - Data types dictionary validation successful.


 - Handling missing values and duplicates...


 - Checking for missing values...


...

   - Found 6 local optima (6 member nodes: 0 plateau-LOs + 6 single-point LOs).


 - Determining global optimum...


   - Global optimum found at index 147 with fitness 99.1723.


Landscape built successfully.



Landscape(kind='default', maximize=True)

## 5. Inspecting the landscape

Let’s first look at what we’ve built. Printing the landscape shows its variables, configurations, improving edges and local optima:

In [6]:
print(landscape)

Landscape(kind='default'): 2 variables, 7504 configurations, 1812150 edges, 6 local optima


For a closer look at individual building-block combinations, we can call `get_data()`. The selected outcome appears as `fitness`; `out_degree` counts directly improving moves, `in_degree` counts incoming moves from lower-scoring neighbours, and `is_lo` indicates local-optimum membership.

In [7]:
landscape.get_data()

,bb,ca,fitness,plateau_id,plateau_size,in_degree,out_degree,is_lo
0,BB01,CA001,21.663005,-1,1,226,257,False
1,BB01,CA002,21.622812,-1,1,227,256,False
2,BB01,CA003,22.786787,-1,1,235,248,False
3,BB01,CA004,26.376782,-1,1,264,219,False
4,BB01,CA005,20.430703,-1,1,221,262,False
...,...,...,...,...,...,...,...,...
7499,BB16,CA465,9.554140,-1,1,444,39,False
7500,BB16,CA466,3.454906,-1,1,351,132,False
7501,BB16,CA467,10.662263,-1,1,452,31,False
7502,BB16,CA468,7.613889,-1,1,441,42,False


We can also summarise the inhibition scores with pandas:

In [8]:
landscape.get_data()["fitness"].describe()

count    7504.000000
mean       15.545357
std        19.983758
min        -5.618241
25%         2.926512
50%         8.092254
75%        20.256774
max        99.172273
Name: fitness, dtype: float64

To inspect the combination with the highest mean mUSP18 inhibition, we can use the global-optimum node index:

In [9]:
landscape[landscape.go_index]

{'bb': 'BB01',
 'ca': 'CA148',
 'fitness': np.float64(99.17227342848624),
 'plateau_id': -1,
 'plateau_size': 1,
 'in_degree': 483,
 'out_degree': 0,
 'is_lo': True}

## 6. Analysing the landscape

Next, we’ll count local optima, measure ruggedness, examine interactions and assess the trend toward the best observed configuration.

In [10]:
from graphfla import analysis

### 6.1 Number of local optima

We can start with `landscape.n_lo`. A local optimum has no improving route out of its immediate neighbourhood, including neutral moves within a plateau. Each connected neutral optimum plateau counts once. Multiple local optima mean that successive improvements can lead to different endpoints.

In [11]:
print(f"Number of local optima: {landscape.n_lo}")

Number of local optima: 6


### 6.2 Roughness-to-slope ratio

For a view of ruggedness, we can calculate `r_s_ratio`. It compares deviations from an additive model with the average magnitude of its coefficients. Larger values indicate more residual roughness relative to the additive trend; a nearly zero fitted slope can give an infinite ratio.


Here, an additive model assigns independent contributions to the amine and acid choices. The ratio describes the variation in assay response that these contributions do not explain.

In [12]:
roughness_to_slope = analysis.r_s_ratio(landscape)
print(f"Roughness-to-slope ratio: {roughness_to_slope:.4f}")

Roughness-to-slope ratio: 1.3094


### 6.3 Interactions between chemical building blocks

Replacing an acid fragment may improve inhibition with one amine but reduce it with another. We can examine this context dependence using `classify_epistasis()`. Here, *epistasis* describes interactions in the measured response to building-block choices.

We can explore these patterns with `classify_epistasis()`, which classifies four-configuration graph motifs:

| Category | Interpretation |
|---|---|
| `magnitude` | Changes retain their direction of effect across contexts. |
| `sign` | One change reverses its direction of effect across contexts. |
| `reciprocal_sign` | Both changes reverse their direction of effect across contexts. |

The output gives their proportions among the detected motifs. The `positive` and `negative` fields form a separate partition.


For this large library, we sample motifs with `sample_cut_prob=0.9` to keep the calculation quick. The proportions are estimates; the fixed seed makes this sample repeatable.

In [13]:
interactions = analysis.classify_epistasis(landscape, sample_cut_prob=0.9, seed=42)
interactions

EpistasisClassification(magnitude=0.5354497354497354, sign=0.3682539682539683, reciprocal_sign=0.0962962962962963, positive=0.8797687861271676, negative=0.12023121387283242)

### 6.4 Fitness-distance correlation

Finally, we can use `fdc` to check whether higher-scoring configurations tend to lie closer to the best observed configuration—a simple view of navigability.


Distance counts how many building blocks differ from the highest-scoring combination. It does not measure molecular fingerprint similarity or a synthetic reaction pathway.

We use Spearman correlation to compare ranks. A negative value means higher scores tend to occur nearer the optimum; a value near zero indicates little monotonic association. This trend does not guarantee an improving path from every configuration.

In [14]:
fitness_distance_r = analysis.fdc(landscape, method="spearman")
print(f"Fitness-distance correlation: {fitness_distance_r:.4f}")

 - Calculating distances to global optimum using mixed_distance...


   - Distances to GO calculated and added as node attribute 'dist_go'.


Fitness-distance correlation: -0.2367


## 7. Running several analyses together

Once we’ve tried the metrics individually, we can collect them with `analysis.profile()`. Here, we combine the ruggedness measure and fitness-distance correlation in a pandas Series. Setting `on_error="raise"` displays errors immediately.

In [15]:
analysis.profile(
    landscape,
    include=["r_s_ratio", "fdc"],
    on_error="raise",
    progress=False,
)

r_s_ratio    1.309404
fdc         -0.236737
dtype: float64

## 8. Analysis reference

For further exploration, we can choose from the **30 analysis functions** below. Each takes a built `landscape` as its first argument. In these functions, *fitness* refers to the outcome selected as `f`.

Walsh–Hadamard and higher-order regression are included for reference. Their cost grows rapidly with the number of variables and levels, so we leave them out of the worked examples.

| Group | Function | Result |
| --- | --- | --- |
| Fitness | `fitness_distribution` | Dictionary of distribution summaries, including skewness, kurtosis, CV and Cauchy location. |
| Fitness | `fitness_effect_distribution` | List of signed effects for one mutation across matched backgrounds. |
| Ruggedness | `local_optima_ratio` | Local-optimum count divided by configuration count. |
| Ruggedness | `r_s_ratio` | Additive-model residual roughness divided by mean absolute coefficient. |
| Ruggedness | `autocorrelation` | Fitness autocorrelation along sampled walks. |
| Ruggedness | `gradient_intensity` | Mean absolute change on improving edges divided by mean fitness. |
| Correlation | `fdc` | Fitness-distance correlation: Spearman or Pearson. |
| Correlation | `neighbor_fitness_correlation` | Correlation between fitness and mean neighbour fitness: Pearson, Spearman or Kendall. |
| Correlation | `basin_fitness_correlation` | Correlation between peak fitness and greedy basin size: Spearman or Pearson. |
| Correlation | `fitness_flattening_index` | Trend in successive fitness gains along greedy walks: Spearman or Pearson. |
| Navigability | `global_optima_accessibility` | Fraction of configurations with an improving route to the global optimum. |
| Navigability | `local_optima_accessibility` | DataFrame of accessibility values for specified local optima. |
| Navigability | `mean_distance_to_global_optimum` | Mean sequence/configuration distance to the global optimum. |
| Navigability | `mean_distance_to_local_optima` | DataFrame of mean sequence/configuration distances to specified local optima. |
| Navigability | `mean_path_length_to_global_optimum` | Mean shortest graph-path length, excluding unreachable starts. |
| Navigability | `mean_path_length_to_local_optima` | DataFrame of shortest-path summaries, excluding unreachable starts. `lo=None` selects the global optimum. |
| Robustness | `neutrality` | Fraction of neighbour pairs within the fitness-difference threshold. |
| Robustness | `evolvability_enhancing_fraction` | Fraction of all directed neighbour pairs classified as EE using non-focal neighbourhoods and BH-corrected tests. |
| Robustness | `single_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at one position; positive or negative tests. |
| Robustness | `all_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at all positions. |
| Epistasis | `idiosyncratic_index` | Normalized spread of one mutation’s effects across backgrounds. |
| Epistasis | `global_idiosyncratic_index` | Mean idiosyncratic index across eligible mutations. |
| Epistasis | `gamma` | Correlation of mutation effects across matched two-site backgrounds. |
| Epistasis | `gamma_star` | Consistency of mutation-effect signs across backgrounds. |
| Epistasis | `diminishing_returns_index` | Association of background fitness with mean beneficial gain: Pearson, Spearman or regression slope. |
| Epistasis | `increasing_costs_index` | Association of background fitness with mean deleterious cost: Pearson, Spearman or regression slope. |
| Epistasis | `higher_order_epistasis` | Per-order cumulative R² and incremental fit gains; accepts an existing W-H result without refitting. |
| Epistasis | `walsh_hadamard` | Coefficients and an order summary with fit gains and the fitted-model variance spectrum. |
| Epistasis | `classify_epistasis` | Object with magnitude, sign, reciprocal-sign, positive and negative epistasis proportions. |
| Epistasis | `extradimensional_bypass` | Object with bypass proportion, mean length and motif counts. |

### Data source

Kooij R., Pol V., Gan J. et al. (2025). [*High-Throughput Synthesis and Screening of a Cyanimide Library Identifies Selective Inhibitors of ISG15-Specific Protease mUSP18*. Angewandte Chemie International Edition 64, e202510941](https://doi.org/10.1002/anie.202510941).

The repository CSV contains the primary-screen measurements from the authors’ supplementary workbook. Repeated measurements are averaged by building-block pair before landscape construction.